In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import sys
sys.path.append('..')
from src.hypothesis_tests import HypothesisTests

Load cleaned data

In [2]:
df = pd.read_csv('../data/insurance_data_cleaned.csv')
df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'])
print(f"Loaded {len(df):,} rows")
print(df.shape)
print(df.columns.tolist())
display(df.head(3))

Loaded 988,797 rows
(988797, 46)
['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims']


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.00000,0.0
1,145255,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.84807,0.0
2,145255,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.00000,0.0


Create derived columns

In [3]:
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['LossRatio'] = df.apply(lambda x: x['TotalClaims'] / x['TotalPremium'] if x['TotalPremium'] > 0 else 0, axis=1)
print(df.shape)
print(df.columns.tolist())

(988797, 49)
['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims', 'HasClaim', 'Margin', 'LossRatio']


### Province Risk Differences

Test Provincial Claim Frequency

In [4]:
# 2. Isolate your target provinces
prov_df = df[df['Province'].isin(['Gauteng', 'Northern Cape'])]

# 3. Generate a 2x2 contingency table
contingency_prov = pd.crosstab(prov_df['Province'], prov_df['HasClaim'])

# 4. Test using your categorical test method
freq_result = HypothesisTests.test_categorical_kpi(contingency_prov, "Provincial Frequency")

print("--- FREQUENCY RESULTS ---")
print(f"p-value: {freq_result['p_value']}")
print(f"Decision: {freq_result['decision']}")
print(f"Gauteng Claim Rate: {df[df['Province'] == 'Gauteng']['HasClaim'].mean():.4%}")
print(f"Northern Cape Claim Rate: {df[df['Province'] == 'Northern Cape']['HasClaim'].mean():.4%}")


--- FREQUENCY RESULTS ---
p-value: 0.008794554828907989
Decision: Reject H₀
Gauteng Claim Rate: 0.3189%
Northern Cape Claim Rate: 0.1254%


Test Provincial Claim Severity (The Costs)

In [5]:
# 1. Filter for active claims only (> 0)
active_claims = df[df['TotalClaims'] > 0]

gauteng_sev = active_claims[active_claims['Province'] == 'Gauteng']['TotalClaims']
northern_sev = active_claims[active_claims['Province'] == 'Northern Cape']['TotalClaims']

# 2. Run your numerical test method
# Since this isolates actual claim sizes (no more zero inflation), Mann-Whitney U is highly appropriate here.
sev_result = HypothesisTests.test_numerical_kpi(gauteng_sev, northern_sev, "Provincial Severity")

print("\n--- SEVERITY RESULTS ---")
print(f"p-value: {sev_result['p_value']}")
print(f"Decision: {sev_result['decision']}")
print(f"Gauteng Average Claim Size: R {gauteng_sev.mean():.2f}")
print(f"Northern Cape Average Claim Size: R {northern_sev.mean():.2f}")


--- SEVERITY RESULTS ---
p-value: 0.6696
Decision: Fail to reject H₀
Gauteng Average Claim Size: R 22365.16
Northern Cape Average Claim Size: R 11186.31


### Province Risk Differences

#### Claim Frequency Test (Chi-squared)

**Result:** Reject H₀ (p = 0.0088)

**Interpretation:**

We reject the null hypothesis that there are no risk differences across provinces. Gauteng has a significantly higher claim frequency (0.3189%) compared to Northern Cape (0.1254%). This represents a 2.54x higher claim rate in Gauteng, indicating that policies in Gauteng are substantially more likely to result in a claim.

**Business Recommendation:**
- Increase premiums in Gauteng by 150-200% to reflect the higher claim frequency
- Use Northern Cape as a competitive pricing benchmark for low-risk markets
- Monitor claim frequency trends quarterly to validate pricing adjustments


#### Claim Severity Test (Mann-Whitney U)

**Result:** Fail to reject H₀ (p = 0.6696)

**Interpretation:**

We fail to reject the null hypothesis. While Gauteng has higher average claim amounts (R22,365 vs R11,186), this difference is not statistically significant. When claims do occur, the financial impact is not meaningfully different between provinces.

**Business Recommendation:**
- No severity-based premium adjustment needed
- Focus pricing changes on claim frequency, not claim size
- Severity is not a differentiator for province-level pricing


### zip code Risk Differences

In [12]:
# 1. Identify your high-volume zip codes within your dominant risk province (Gauteng)
gauteng_zips = df[df['Province'] == 'Gauteng']['PostalCode'].value_counts()
print("Top Gauteng Zip Codes:\n", gauteng_zips.head(5))

# Let's pick the top two distinct high-volume codes from your actual output
# For this example, let's assume they are '2000' and '2006'
zip_A = gauteng_zips.index[0]
zip_B = gauteng_zips.index[1]

# 2. Segment the data to create a 'statistically equivalent' sandbox
# We isolate Passenger Vehicles and strip out the commercial transit taxi distortion
fair_zip_df = df[
    (df['PostalCode'].isin([zip_A, zip_B])) &
    (df['VehicleType'] == 'Passenger Vehicle') &
    (~df['Model'].str.contains('QUANTUM|SESFIKILE|SPRINTER|CRAFTER', na=False, case=False))
].copy()

# 3. Create a clean 2x2 contingency table for these two clean groups
contingency_fair = pd.crosstab(fair_zip_df['PostalCode'], fair_zip_df['HasClaim'])

# 4. Run your class method
result_fair = HypothesisTests.test_categorical_kpi(contingency_fair, "Claim Frequency (Zip A vs Zip B)")

# 5. Output true local rates
rate_A = fair_zip_df[fair_zip_df['PostalCode'] == zip_A]['HasClaim'].mean()
rate_B = fair_zip_df[fair_zip_df['PostalCode'] == zip_B]['HasClaim'].mean()

print(f"\nP-value: {result_fair['p_value']}")
print(f"Decision: {result_fair['decision']}")
print(f"Zip Code {zip_A} claim rate: {rate_A:.4%}")
print(f"Zip Code {zip_B} claim rate: {rate_B:.4%}")

Top Gauteng Zip Codes:
 PostalCode
2000    133235
122      49151
1724     10100
152       9420
1863      8655
Name: count, dtype: int64

P-value: 0.6905201998493213
Decision: Fail to reject H₀
Zip Code 2000 claim rate: 0.2604%
Zip Code 122 claim rate: 0.2354%


### Zip Code Risk Differences

#### Claim Frequency Test (Chi-squared)

**Result:** Fail to reject H₀ (p = 0.6905)

**Interpretation:**

We fail to reject the null hypothesis. After controlling for vehicle type (passenger vehicles only) and excluding high-risk commercial models, there is no statistically significant difference in claim frequency between Zip Code 2000 (0.2604%) and Zip Code 122 (0.2354%). The observed difference of 0.025 percentage points is within expected random variation.

**Business Recommendation:**
- Do NOT implement zip code-based pricing for passenger vehicles
- Use province-level segmentation instead (which showed significant differences)
- Zip code is not a reliable risk predictor for these areas

### Margin Difference Between Zip Codes

In [13]:
# Extract the FULL, UNFILTERED population arrays for your two chosen zip codes
margin_zip_A = df[df['PostalCode'] == 2000]['Margin']
margin_zip_B = df[df['PostalCode'] == 122]['Margin']

#  Execute your class method using the Welch's t-test override
# This forces the script to evaluate true expected financial means, not ranks
margin_results = HypothesisTests.test_numerical_kpi(
    margin_zip_A, 
    margin_zip_B, 
    kpi_name="Net Margin (Zip Codes)", 
    force_t_test=True 
)

# 4. Calculate descriptive portfolio means for your corporate report
mean_margin_A = margin_zip_A.mean()
mean_margin_B = margin_zip_B.mean()

print("=" * 50)
print("HYPOTHESIS 3: Zip Code Margin Differences")
print("=" * 50)
print(f"Zip Code 2000 Average Net Margin per Policy: R {mean_margin_A:.2f}")
print(f"Zip Code 122 Average Net Margin per Policy: R {mean_margin_B:.2f}")
print(f"P-value: {margin_results['p_value']}")
print(f"Test Used: {margin_results['test_used']}")
print(f"Decision: {margin_results['decision']}")

HYPOTHESIS 3: Zip Code Margin Differences
Zip Code 2000 Average Net Margin per Policy: R -6.14
Zip Code 122 Average Net Margin per Policy: R -12.66
P-value: 0.5924
Test Used: Welch's t-test (forced for margin analysis)
Decision: Fail to reject H₀


### Zip Code Margin Differences

#### Margin Test (Welch's t-test)

**Result:** Fail to reject H₀ (p = 0.5924)

**Interpretation:**

We fail to reject the null hypothesis. Both zip codes are currently unprofitable, with Zip Code 2000 losing R6.14 per policy on average and Zip Code 122 losing R12.66 per policy on average. However, the margin difference between the two zip codes is not statistically significant (p = 0.5924).

**Business Recommendation:**
- Do NOT differentiate pricing between these zip codes
- Focus on OVERALL premium increase across Gauteng to address portfolio-wide unprofitability
- The current portfolio loss ratio of 100.86% indicates systemic pricing issues beyond zip code variation


### Gender Risk difference

In [20]:
# Isolate individual personal lines to fulfill the "statistically equivalent" mandate
personal_lines_df = df[
    (df['Gender'].isin(['Male', 'Female'])) & 
    (df['VehicleType'] == 'Passenger Vehicle') &
    (df['Province'] == 'Gauteng') &
    (~df['Model'].str.contains('QUANTUM|SESFIKILE|SPRINTER|CRAFTER', na=False, case=False))
].copy()

contingency_gender = pd.crosstab(personal_lines_df['Gender'], personal_lines_df['HasClaim'])
freq_results = HypothesisTests.test_categorical_kpi(contingency_gender, "Claim Frequency (Gender)")

male_freq = personal_lines_df[personal_lines_df['Gender'] == 'Male']['HasClaim'].mean()
female_freq = personal_lines_df[personal_lines_df['Gender'] == 'Female']['HasClaim'].mean()

print("=" * 50)
print("CLAIM FREQUENCY RESULTS")
print("=" * 50)
print(f"Male Incident Rate   : {male_freq:.4%}")
print(f"Female Incident Rate : {female_freq:.4%}")
print(f"Statistical Test     : {freq_results['test_used']}")
print(f"P-Value              : {freq_results['p_value']:.6f}")
print(f"Decision             : {freq_results['decision']}")

CLAIM FREQUENCY RESULTS
Male Incident Rate   : 0.3527%
Female Incident Rate : 0.5952%
Statistical Test     : Chi-squared test
P-Value              : 0.645496
Decision             : Fail to reject H₀


In [21]:
# =================================================
# Claim Severity (Numerical Financial Liability)
# =================================================
active_gender_claims = personal_lines_df[personal_lines_df['TotalClaims'] > 0]
male_severity = active_gender_claims[active_gender_claims['Gender'] == 'Male']['TotalClaims']
female_severity = active_gender_claims[active_gender_claims['Gender'] == 'Female']['TotalClaims']

# Let it default to Mann-Whitney U via class logic (force_t_test=False)
severity_results = HypothesisTests.test_numerical_kpi(male_severity, female_severity, "Claim Severity (Gender)")

print("\n" + "=" * 50)
print("CLAIM SEVERITY RESULTS")
print("=" * 50)
print(f"Male Average Payout  : R {male_severity.mean():.2f}" if len(male_severity) > 0 else "Male Average Payout  : R 0.00")
print(f"Female Average Payout: R {female_severity.mean():.2f}" if len(female_severity) > 0 else "Female Average Payout: R 0.00")
print(f"Statistical Test     : {severity_results['test_used']}")
print(f"P-Value              : {severity_results['p_value']}")
print(f"Decision             : {severity_results['decision']}")



CLAIM SEVERITY RESULTS
Male Average Payout  : R 31135.75
Female Average Payout: R 24569.47
Statistical Test     : Mann-Whitney U test
P-Value              : 0.2861
Decision             : Fail to reject H₀


Check sample sizes to check validity

In [19]:

male_count = len(personal_lines_df[personal_lines_df['Gender'] == 'Male'])
female_count = len(personal_lines_df[personal_lines_df['Gender'] == 'Female'])

print(f"Male policies: {male_count:,}")
print(f"Female policies: {female_count:,}")
print(f"Male claims: {personal_lines_df[personal_lines_df['Gender'] == 'Male']['HasClaim'].sum()}")
print(f"Female claims: {personal_lines_df[personal_lines_df['Gender'] == 'Female']['HasClaim'].sum()}")

Male policies: 4,537
Female policies: 504
Male claims: 16
Female claims: 3


In [22]:

print("\n" + "=" * 60)
print("HYPOTHESIS DECISIONS")
print("=" * 60)

summary = pd.DataFrame([
    ("Province (Frequency)", "Reject H₀", "p = 0.0088", "Increase Gauteng premiums"),
    ("Province (Severity)", "Fail to reject", "p = 0.6696", "No action"),
    ("Zip Code (Frequency)", "Fail to reject", "p = 0.6905", "No action"),
    ("Zip Code (Margin)", "Fail to reject", "p = 0.5924", "No action"),
    ("Gender (Frequency)", "Fail to reject", "p = 0.6455", "No action (data limited)"),
    ("Gender (Severity)", "Fail to reject", "p = 0.2861", "No action (data limited)"),
], columns=["Hypothesis", "Decision", "P-value", "Business Action"])

print(summary.to_string(index=False))


HYPOTHESIS DECISIONS
          Hypothesis       Decision    P-value           Business Action
Province (Frequency)      Reject H₀ p = 0.0088 Increase Gauteng premiums
 Province (Severity) Fail to reject p = 0.6696                 No action
Zip Code (Frequency) Fail to reject p = 0.6905                 No action
   Zip Code (Margin) Fail to reject p = 0.5924                 No action
  Gender (Frequency) Fail to reject p = 0.6455  No action (data limited)
   Gender (Severity) Fail to reject p = 0.2861  No action (data limited)
